In [2]:
# package import
import numpy as np
from numpy import arange
import pandas as pd
from pandas import read_csv
from pandas import set_option
from pandas.plotting import scatter_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score

import ipywidgets as widgets

In [3]:
mse_df = pd.read_csv("mse_results.csv")

In [5]:
k_values = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25]

In [6]:
colors = {"all_61": "C0", "starting_model": "C1", "expanded_model": "C2",
          "compact_model": "C3", "compact_plus_model": "C4"}
styles = {"uniform": "-", "euclidean": "--", "manhattan": "-.", "chebyshev": ":"}

def plot_mse(models, settings):
    if isinstance(models, str):   models = [models]       # single choice -> list
    if isinstance(settings, str): settings = [settings]
    plt.figure(figsize=(10, 6))
    for name in models:
        for setting in settings:
            d = mse_df[(mse_df["model"] == name) & (mse_df["setting"] == setting)]
            plt.plot(d["k"], d["cv_mse"], styles[setting], marker="o", markersize=3,
                     color=colors[name], label="%s (%s)" % (name, setting))
    plt.xlabel("Number of Neighbors, K")
    plt.ylabel("CV MSE (lower is better)")
    plt.xticks(k_values)
    plt.grid(alpha=0.3)
    if models and settings:
        plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


# ----- checkboxes -----
model_boxes   = {m: widgets.Checkbox(value=(m == "compact_model"), description=m) for m in colors}
setting_boxes = {s: widgets.Checkbox(value=(s == "uniform"), description=s) for s in styles}

def plot_from_boxes(**checked):
    models   = [m for m in colors if checked[m]]
    settings = [s for s in styles if checked[s]]
    plot_mse(models, settings)

out = widgets.interactive_output(plot_from_boxes, {**model_boxes, **setting_boxes})

display(widgets.HBox([
    widgets.VBox([widgets.Label("Models")]   + list(model_boxes.values())),
    widgets.VBox([widgets.Label("Settings")] + list(setting_boxes.values())),
]), out)

Output()

In [7]:
balacc_df = pd.read_csv("balacc_results.csv")

In [8]:
colors = {"all_61": "C0", "starting_model": "C1", "expanded_model": "C2",
          "compact_model": "C3", "compact_plus_model": "C4"}
styles = {"uniform": "-", "euclidean": "--", "manhattan": "-.", "chebyshev": ":"}

def plot_balacc(models, settings):
    if isinstance(models, str):   models = [models]       # single choice -> list
    if isinstance(settings, str): settings = [settings]
    plt.figure(figsize=(10, 6))
    for name in models:
        for setting in settings:
            d = balacc_df[(balacc_df["model"] == name) & (balacc_df["setting"] == setting)]
            plt.plot(d["k"], d["cv_balacc"], styles[setting], marker="o", markersize=3,
                     color=colors[name], label="%s (%s)" % (name, setting))

    plt.xlabel("Number of Neighbors, K")
    plt.ylabel("CV balanced accuracy (higher is better)")
    plt.xticks(k_values)
    plt.grid(alpha=0.3)
    if models and settings:
        plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


# ----- checkboxes -----
b_model_boxes   = {m: widgets.Checkbox(value=(m == "compact_model"), description=m) for m in colors}
b_setting_boxes = {s: widgets.Checkbox(value=(s == "uniform"), description=s) for s in styles}

def plot_balacc_from_boxes(**checked):
    models   = [m for m in colors if checked[m]]
    settings = [s for s in styles if checked[s]]
    plot_balacc(models, settings)

b_out = widgets.interactive_output(plot_balacc_from_boxes, {**b_model_boxes, **b_setting_boxes})

display(widgets.HBox([
    widgets.VBox([widgets.Label("Models")]   + list(b_model_boxes.values())),
    widgets.VBox([widgets.Label("Settings")] + list(b_setting_boxes.values())),
]), b_out)

Output()

In [9]:
thr_df = pd.read_csv("threshold_results.csv")

In [10]:
thr_03 = thr_df[np.isclose(thr_df["threshold"], 0.30)]          # only threshold = 0.30

thr_colors = {"all_61": "C0", "starting_model": "C1", "expanded_model": "C2",
              "compact_model": "C3", "compact_plus_model": "C4", "compact_ultra_model": "C5"}
thr_styles = {"uniform": "-", "euclidean": "--", "manhattan": "-.", "chebyshev": ":"}

thr_models_avail   = [m for m in thr_colors if m in thr_03["model"].unique()]
thr_settings_avail = [s for s in thr_styles if s in thr_03["setting"].unique()]

def thr_plot_03(thr_models, thr_sets):
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for thr_m in thr_models:
        for thr_s in thr_sets:
            thr_dd = thr_03[(thr_03["model"] == thr_m) & (thr_03["setting"] == thr_s)].sort_values("k")
            lbl = "%s (%s)" % (thr_m, thr_s)
            axes[0].plot(thr_dd["k"], thr_dd["error_rate"], thr_styles[thr_s], marker="o",
                         markersize=3, color=thr_colors[thr_m], label=lbl)
            axes[1].plot(thr_dd["k"], thr_dd["bal_acc"], thr_styles[thr_s], marker="o",
                         markersize=3, color=thr_colors[thr_m], label=lbl)

    axes[0].set_title("Threshold = 0.30: CV MSE by k")
    axes[0].set_ylabel("CV MSE")
    axes[1].set_title("Threshold = 0.30: CV balanced accuracy by k")
    axes[1].set_ylabel("CV balanced accuracy")
    axes[1].set_xlabel("Number of Neighbors, K")
    for ax in axes:
        ax.set_xticks(sorted(thr_03["k"].unique()))
        ax.grid(alpha=0.3)
    if thr_models and thr_sets:
        axes[0].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("threshold_030_plot.png", dpi=150, bbox_inches="tight")
    plt.show()

# ----- checkboxes -----
thr_model_boxes   = {m: widgets.Checkbox(value=(m == "compact_plus_model"), description=m)
                     for m in thr_models_avail}
thr_setting_boxes = {s: widgets.Checkbox(value=(s == "euclidean"), description=s)
                     for s in thr_settings_avail}

def thr_plot_from_boxes(**thr_checked):
    thr_plot_03([m for m in thr_models_avail if thr_checked[m]],
                [s for s in thr_settings_avail if thr_checked[s]])

thr_out = widgets.interactive_output(thr_plot_from_boxes, {**thr_model_boxes, **thr_setting_boxes})

display(widgets.HBox([
    widgets.VBox([widgets.Label("Models")]   + list(thr_model_boxes.values())),
    widgets.VBox([widgets.Label("Settings")] + list(thr_setting_boxes.values())),
]), thr_out)

Output()